In [ ]:
# ============================================================
# CELL 1: Install all required libraries
# Run this first. Takes ~2 minutes. Wait for the checkmark.
# ============================================================

# Unsloth: makes Qwen2.5 training 2x faster + fits on free T4
!pip -q install unsloth

# Specific versions that work together without conflicts
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2

# datasets: for loading our JSONL files cleanly
!pip -q install -U datasets

print("✓ All libraries installed")

In [ ]:
# ============================================================
# CELL 2: Import everything we need
# ============================================================

import os
import gc
import re
import time
import json
import warnings
from typing import List

warnings.filterwarnings("ignore")

import torch
from datasets import Dataset, load_dataset

# Unsloth's fast model loader — handles 4-bit QLoRA automatically
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

# ── GPU check ──────────────────────────────────────────────
# If this fails → Runtime → Change runtime type → T4 GPU
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✓ GPU: {gpu_name}")
print(f"✓ VRAM available: {vram_gb:.1f} GB")
print(f"✓ BF16 supported: {is_bfloat16_supported()}")

In [ ]:
# ============================================================
# CELL 3: All configuration in one place
# This is the ONLY cell you need to change if you want to
# experiment with different settings.
# ============================================================

# ── Model ──────────────────────────────────────────────────
# Qwen2.5-1.5B-Instruct: best small instruction model on Unsloth.
# -bnb-4bit suffix means it loads in 4-bit (QLoRA) — fits on free T4.
BASE_MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

# Maximum tokens per training sequence.
# 1024 gives room for your Q&A pairs (avg ~250 tokens each).
# 512 would be too tight and silently truncate long answers.
MAX_SEQ_LENGTH = 1024

# Random seed — keeps results reproducible across runs
SEED = 42

# ── LoRA / QLoRA Config ────────────────────────────────────
# r=32: rank of the low-rank adapter matrices.
# Higher r = more trainable parameters = more learning capacity.
# r=16 is minimum acceptable. r=32 is the competition-grade sweet spot
# for a 1.5B model with ~175 training examples.
# (r=64 would risk overfitting on our dataset size)
LORA_R = 32

# alpha controls the scaling of LoRA updates.
# Rule of thumb: alpha = 2 × r. So alpha=64 for r=32.
# This keeps the update magnitude in a stable range.
LORA_ALPHA = 64

# Small dropout prevents overfitting on the LoRA matrices.
# 0.05 is standard for small datasets like ours.
LORA_DROPOUT = 0.05

# ── Training Batch Config ──────────────────────────────────
# per_device_train_batch_size × gradient_accumulation_steps = effective batch size
# 2 × 4 = 8 effective batch size — good balance of stability and speed on T4
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 4
# Effective batch = 2 × 4 = 8 examples per weight update

# Warmup: gradually increases LR from 0 to target over first N steps.
# Prevents large gradient spikes at the start of training.
WARMUP_STEPS  = 5
LOGGING_STEPS = 5  # print loss every 5 steps

# ── Epochs per Stage ───────────────────────────────────────
# Stage 1: 3 epochs over 80 paragraphs. With packing=True, these become
# ~27 packed sequences. 3 epochs = model sees each paragraph 3 times.
# Enough to shift domain knowledge without overfitting.
STAGE1_EPOCHS = 3

# Stage 2: 3 epochs over 175 Q&A pairs.
# 3 epochs is the standard for SFT on small instruction datasets.
# More epochs → better training signal. 5+ epochs → overfitting risk.
STAGE2_EPOCHS = 3

# Stage 3: 1 epoch over 77 preference pairs.
# DPO is VERY sensitive to over-training. 1 epoch is the industry standard.
# More than 1 epoch on DPO almost always hurts answer quality.
STAGE3_EPOCHS = 1

# ── Learning Rates ─────────────────────────────────────────
# Stage 1: Higher LR is fine — we're doing domain adaptation, not instruction tuning.
# The model just needs to shift its vocabulary distributions.
STAGE1_LR = 2e-4

# Stage 2: Same LR as Stage 1 since we start from a fresh LoRA on the merged model.
# The merged model IS Stage 1 — Stage 2 LoRA starts from zero.
STAGE2_LR = 2e-4

# Stage 3: Much lower LR for DPO. Preference alignment is delicate —
# too high an LR collapses the model to one mode (all answers look the same).
STAGE3_LR = 5e-5

# DPO beta: controls how strongly we push away from rejected responses.
# 0.1 is the standard starting value. Lower = softer preference signal.
DPO_BETA = 0.1

# ── Minimum paragraph length for corpus ────────────────────
# Paragraphs shorter than this are skipped in Stage 1 data loading.
# 80 chars = roughly 1-2 sentences. Keeps noisy short snippets out.
MIN_CHARS_PER_PARAGRAPH = 80

# ── File paths ─────────────────────────────────────────────
# Upload these 3 files to Colab before running Stage cells.
NON_INSTRUCTION_PATH = "/content/non_instruction_corpus.txt"
INSTRUCTION_PATH     = "/content/instruction_dataset.jsonl"
PREFERENCE_PATH      = "/content/preference_dataset.jsonl"

# ── Output directories ─────────────────────────────────────
OUTPUT_ROOT       = "/content/placement_assistant_outputs"
STAGE1_ADAPTER    = f"{OUTPUT_ROOT}/stage1_adapter"
STAGE1_MERGED     = f"{OUTPUT_ROOT}/stage1_merged"
STAGE2_ADAPTER    = f"{OUTPUT_ROOT}/stage2_adapter"
STAGE2_MERGED     = f"{OUTPUT_ROOT}/stage2_merged"
STAGE3_ADAPTER    = f"{OUTPUT_ROOT}/stage3_adapter"
FINAL_MERGED      = f"{OUTPUT_ROOT}/final_merged_model"

for d in [OUTPUT_ROOT, STAGE1_ADAPTER, STAGE1_MERGED,
          STAGE2_ADAPTER, STAGE2_MERGED, STAGE3_ADAPTER, FINAL_MERGED]:
    os.makedirs(d, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Model      : {BASE_MODEL_NAME}")
print(f"  LoRA rank  : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Eff. batch : {BATCH_SIZE} × {GRAD_ACCUM_STEPS} = {BATCH_SIZE*GRAD_ACCUM_STEPS}")
print(f"  Epochs     : Stage1={STAGE1_EPOCHS}, Stage2={STAGE2_EPOCHS}, Stage3={STAGE3_EPOCHS}")
print(f"  Output dir : {OUTPUT_ROOT}")

In [ ]:
# ============================================================
# CELL 4: Helper functions used across all 3 stages
# ============================================================

def clear_gpu_memory():
    """
    Frees GPU memory between stages.
    Without this, loading Stage 2 model while Stage 1 is still
    in memory will cause an OOM (out-of-memory) crash on T4.
    """
    gc.collect()
    torch.cuda.empty_cache()
    print("  GPU memory cleared")


def train_and_measure(trainer, stage_name: str):
    """
    Runs training and prints time + VRAM usage.
    The VRAM peak numbers go into your README — shows you understand
    hardware constraints, which is a differentiator in your report.
    """
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    elapsed   = round(time.time() - start, 1)
    peak_alloc = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    peak_res   = round(torch.cuda.max_memory_reserved()  / 1024**3, 2)

    print(f"\n{'='*50}")
    print(f"{stage_name} — TRAINING COMPLETE")
    print(f"  Training time    : {elapsed}s ({elapsed/60:.1f} min)")
    print(f"  Final loss       : {result.training_loss:.4f}")
    print(f"  Peak VRAM used   : {peak_alloc} GB allocated / {peak_res} GB reserved")
    print(f"{'='*50}")
    return result


def load_model_with_lora(model_path: str):
    """
    Loads a model (base or previously merged) in 4-bit QLoRA
    and attaches a fresh LoRA adapter for training.

    This is called 3 times:
      Stage 1 → loads BASE_MODEL_NAME
      Stage 2 → loads STAGE1_MERGED  (has Stage 1 knowledge baked in)
      Stage 3 → loads STAGE2_MERGED  (has Stage 1 + Stage 2 knowledge baked in)

    Each time, a FRESH LoRA adapter is attached — we're not stacking adapters,
    we're training a new adapter on top of an increasingly capable base.
    """
    print(f"  Loading: {model_path}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name    = model_path,
        max_seq_length= MAX_SEQ_LENGTH,
        dtype         = None,        # auto-detect: BF16 if supported, else FP16
        load_in_4bit  = True,        # QLoRA: quantise base to 4-bit to save VRAM
    )

    # Pad token needed for batching — Qwen uses EOS as pad
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # right padding for SFT, left for DPO

    # Attach fresh LoRA adapter
    # target_modules: which weight matrices to add LoRA to.
    # These 7 modules cover all attention + feed-forward layers in Qwen2.5.
    # More modules = more trainable params = better learning but more VRAM.
    model = FastLanguageModel.get_peft_model(
        model,
        r                  = LORA_R,
        lora_alpha         = LORA_ALPHA,
        lora_dropout       = LORA_DROPOUT,
        target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                               "gate_proj", "up_proj", "down_proj"],
        bias               = "none",
        # "unsloth" mode: Unsloth's custom gradient checkpointing.
        # Reduces VRAM by ~30% by recomputing activations on the fly.
        # Slightly slower but essential to fit on free T4.
        use_gradient_checkpointing = "unsloth",
        random_state       = SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Two-step save:
    1. Save LoRA adapter only (small, fast, ~20MB)
    2. Merge adapter INTO base weights and save full model (larger, ~3GB)

    The merged model is what the NEXT stage loads as its base.
    The adapter is kept separately for your GitHub repo (lightweight artifact).
    """
    # Step 1: save adapter only
    print(f"\n  Saving {stage_name} LoRA adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"  ✓ Adapter saved → {adapter_dir}")

    # Step 2: merge adapter weights into base model and save
    # save_method='merged_16bit': saves in FP16 (half precision).
    # Smaller than FP32, still high quality. Standard for deployment.
    print(f"  Merging {stage_name} adapter into base weights...")
    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method = "merged_16bit",
    )
    print(f"  ✓ Merged model saved → {merged_dir}")


def generate_answer(model, tokenizer, question: str, max_new_tokens: int = 200) -> str:
    """
    Runs inference to test the model after each stage.
    We use the same prompt format as training so the model recognises
    the instruction structure it was trained on.

    Safety net: even after the EOS-token fix in Stage 2 training, we
    truncate on common "new turn" markers in case the model ever
    still drifts into inventing a fake next question.
    """
    FastLanguageModel.for_inference(model)

    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            do_sample         = True,
            temperature       = 0.7,
            top_p             = 0.9,
            repetition_penalty= 1.1,
            pad_token_id      = tokenizer.eos_token_id,
            eos_token_id      = tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    text = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # Cut off if the model starts hallucinating a new turn
    for stop_marker in ["### Instruction", "Human:", "\nQ:", "###"]:
        if stop_marker in text:
            text = text.split(stop_marker)[0].strip()

    return text

print("✓ Helper functions defined")

In [ ]:
# ============================================================
# CELL 5: Stage 1 — Load domain corpus
# ============================================================
# Before running this cell: upload non_instruction_corpus.txt to Colab
# (click the Files icon on the left sidebar → upload)
# ============================================================

def load_corpus_dataset(txt_path: str) -> Dataset:
    """
    Loads the plain text corpus for Stage 1 non-instruction fine-tuning.

    Stage 1 does NOT use Q&A format. It just reads raw paragraphs of
    placement domain text. The model learns:
    - Company names and their round structures
    - Domain vocabulary (bar-raiser, NQT, CGPA, PPO, etc.)
    - Writing style of placement advice
    Think of it as the model reading a placement textbook before studying Q&A.
    """
    if not os.path.exists(txt_path):
        raise FileNotFoundError(
            f"File not found: {txt_path}\n"
            "Upload non_instruction_corpus.txt to Colab via the Files sidebar."
        )

    with open(txt_path, "r", encoding="utf-8") as f:
        raw = f.read()

    # Split on double newlines (paragraph boundaries)
    # Filter out anything too short to be meaningful
    paragraphs = [
        p.strip()
        for p in raw.split("\n\n")
        if len(p.strip()) >= MIN_CHARS_PER_PARAGRAPH
    ]

    if len(paragraphs) == 0:
        raise ValueError("No valid paragraphs found. Check your corpus file.")

    print(f"✓ Corpus loaded: {len(paragraphs)} paragraphs")
    print(f"  Avg length   : {sum(len(p) for p in paragraphs)//len(paragraphs)} chars")
    print(f"  Sample (first 150 chars):")
    print(f"  '{paragraphs[0][:150]}...'")

    # Dataset needs a 'text' column — SFTTrainer reads from dataset_text_field='text'
    return Dataset.from_dict({"text": paragraphs})


stage1_dataset = load_corpus_dataset(NON_INSTRUCTION_PATH)
print(f"\n✓ Stage 1 dataset ready: {len(stage1_dataset)} training examples")

In [ ]:
# ============================================================
# CELL 6: Stage 1 — Non-Instruction Fine-Tuning
# Expected time: 8-12 minutes on T4
# Expected final loss: ~1.5 to 2.5 (language modelling loss, lower is better)
# ============================================================
# SCREENSHOT the loss output from this cell — it goes in your README
# as proof of training (this is a required deliverable)
# ============================================================

print("=" * 55)
print("STAGE 1: NON-INSTRUCTION DOMAIN FINE-TUNING")
print("Goal: Teach the model placement domain vocabulary")
print("=" * 55)

# Load fresh base model with LoRA attached
stage1_model, tokenizer = load_model_with_lora(BASE_MODEL_NAME)
FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir   = f"{OUTPUT_ROOT}/stage1_logs",

    # num_train_epochs vs max_steps:
    # We use epochs (not max_steps) so the model sees ALL paragraphs.
    # max_steps=30 (like the original notebook) would stop early
    # and miss half the data. Epochs guarantees full coverage.
    num_train_epochs             = STAGE1_EPOCHS,

    per_device_train_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps  = GRAD_ACCUM_STEPS,
    # Effective batch size = 2 × 4 = 8

    learning_rate  = STAGE1_LR,
    # 2e-4 is Unsloth's recommended LR for domain adaptation.
    # Higher than Stage 2/3 because we want the model to shift
    # its domain distribution significantly.

    warmup_steps   = WARMUP_STEPS,
    logging_steps  = LOGGING_STEPS,

    save_strategy  = "no",      # don't save mid-training checkpoints (saves disk)
    report_to      = "none",    # disable wandb/tensorboard

    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    # T4 uses FP16. A100/L4 use BF16. is_bfloat16_supported() auto-detects.

    optim = "adamw_8bit",
    # 8-bit Adam: same convergence as regular Adam but 4x less VRAM for
    # optimizer states. Essential for fitting Stage 1+2+3 on free T4.

    dataset_text_field = "text",  # column name in our Dataset object
    max_length         = MAX_SEQ_LENGTH,

    # packing=True: packs multiple short paragraphs into one 1024-token sequence.
    # Our paragraphs average ~175 tokens, so ~5 paragraphs fit per sequence.
    # This is a HUGE efficiency win — reduces number of training steps
    # while making sure no GPU time is wasted on padding tokens.
    # NOTE: packing=True is only safe for Stage 1 (raw text).
    # In Stage 2, we turn it OFF to preserve Q&A boundaries.
    packing = True,

    seed = SEED,
)

stage1_trainer = SFTTrainer(
    model            = stage1_model,
    processing_class = tokenizer,
    train_dataset    = stage1_dataset,
    args             = stage1_config,
)

# Train and print metrics
stage1_result = train_and_measure(stage1_trainer, "STAGE 1")

# Quick sanity test — run BEFORE merging
# Expected: model should complete the text in placement-domain style.
# It will NOT answer the question properly yet (that's Stage 2's job).
# As long as it generates coherent placement-related text = Stage 1 worked.
print("\nStage 1 sanity test (text completion, NOT Q&A yet):")
test_output = generate_answer(
    stage1_model, tokenizer,
    "Amazon SDE-1 interview preparation includes",
    max_new_tokens=80
)
print(f"  Output: {test_output}")

# Save adapter + merge into base model
# The merged model becomes Stage 2's starting point
save_and_merge(
    model      = stage1_model,
    tokenizer  = tokenizer,
    adapter_dir= STAGE1_ADAPTER,
    merged_dir = STAGE1_MERGED,
    stage_name = "Stage 1",
)

# CRITICAL: delete model from GPU before loading Stage 2 model
# Without this, you'll get OOM when loading Stage 2
del stage1_trainer
del stage1_model
clear_gpu_memory()

print("\n✓ Stage 1 complete. Stage 1 merged model ready for Stage 2.")
print(f"  Merged model location: {STAGE1_MERGED}")